# 1. Instalación de librerias

In [3]:
!pip install nltk pymongo pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 17.9 MB/s eta 0:00:00


#2. Importar librerias

In [4]:
import requests
import pandas as pd
from datetime import datetime
from pymongo import MongoClient
import os
import json

#3. Conexión a MongoDB

In [8]:
# Leer las credenciales desde el archivo JSON
with open('config.json', 'r') as f:
    config = json.load(f)

# Fecha actual para nombrar archivo y colección
today_str = datetime.today().strftime("%d_%m_%Y")
file_name = f"memecoins_volatility_data_{today_str}"
csv_path = f"{file_name}.csv"

# Configuración de MongoDB
uri = config["mongo_uri"]
client = MongoClient(uri)
db = client["crypto_analysis"]
collection = db[file_name]



#4. Definir los identificadores de las monedas y procesar los datos de CoinGecko

In [6]:
# Diccionario con los IDs de CoinGecko
coins = {
    'pepe': 'pepe',
    'doge': 'dogecoin',
    'shiba': 'shiba-inu'
}

all_volatility = []

for name, coingecko_id in coins.items():
    print(f"📊 Procesando {name.upper()}...")

    url = f'https://api.coingecko.com/api/v3/coins/{coingecko_id}/ohlc'
    params = {
        'vs_currency': 'usd',
        'days': '365'
    }

    response = requests.get(url, params=params)
    data = response.json()

    if not data:
        print(f"No se pudieron obtener datos para {name.upper()}.")
        continue

    # Crear DataFrame
    df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])

    # Convertir timestamp a fecha
    df['date'] = pd.to_datetime(df['timestamp'], unit='ms').dt.date

    # Calcular métricas de volatilidad
    df['range'] = df['high'] - df['low']
    df['pct_change'] = (df['close'] - df['open']) / df['open'] * 100
    df['rel_volatility'] = (df['high'] - df['low']) / ((df['high'] + df['low']) / 2)

    # Redondear resultados
    df['range'] = df['range'].round(10)
    df['pct_change'] = df['pct_change'].round(6)
    df['rel_volatility'] = df['rel_volatility'].round(6)

    # Agregar identificador de moneda
    df['coin'] = name

    final_df = df[['date', 'open', 'high', 'low', 'close', 'range', 'pct_change', 'rel_volatility', 'coin']]
    all_volatility.append(final_df)


📊 Procesando PEPE...
📊 Procesando DOGE...
📊 Procesando SHIBA...


#5. Combinar los datos, guardarlos en CSV y subirlos a MongoDB

In [9]:
# Combinar todo
if not all_volatility:
    print("No se generaron datos.")
else:
    combined_volatility = pd.concat(all_volatility, ignore_index=True)
    combined_volatility['date'] = pd.to_datetime(combined_volatility['date'])

    # Guardar CSV
    combined_volatility.to_csv(csv_path, index=False, sep=';', decimal=',')
    print(f"CSV guardado: {csv_path}")

    # Verificar si la colección ya existe y eliminarla si es necesario
    if file_name in db.list_collection_names():
        db.drop_collection(file_name)
        print(f"Colección '{file_name}' eliminada previamente.")

    # Subir a MongoDB
    collection.insert_many(combined_volatility.to_dict("records"))
    print(f" Datos subidos a MongoDB: colección '{file_name}'")

CSV guardado: memecoins_volatility_data_05_06_2025.csv
 Datos subidos a MongoDB: colección 'memecoins_volatility_data_05_06_2025'
